# 01 — KuaiSearch-Lite preprocessing và embedding

Notebook chạy trên Kaggle, dùng Pandas cho `items_lite`/`users_lite`, Polars streaming cho `rank_lite` và tạo product embedding đầu vào cho RQ-VAE.

## 0. Cấu hình

In [1]:
from pathlib import Path

RAW_ROOT = None
OUTPUT_ROOT = None

VALIDATION_PERCENT = 10
SEED = 2026
EMBEDDING_MODEL = "jinaai/jina-embeddings-v5-text-nano-clustering"
EMBEDDING_DIM = 256
EMBEDDING_BATCH_SIZE = 128

## 1. Import

In [2]:
import json
import os
import shutil
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "huggingface_hub>=0.27", "pyarrow>=14", "polars>=1.39",
    "sentence-transformers>=5.2.0", "transformers>=5.1.0", "peft>=0.15.2",
])

import numpy as np
import pandas as pd
import polars as pl
import torch
from huggingface_hub import snapshot_download
from sentence_transformers import SentenceTransformer

print("Pandas:", pd.__version__)
print("Polars:", pl.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 847.1/847.1 kB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.3/611.3 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 107.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 34.6 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
cudf-polars-cu12 26.2.1 requires polars<1.36,>=1.30, but you have polars 1.43.2 which is incompatible.


Pandas: 2.3.3
Polars: 1.43.2
GPU: Tesla T4


## 2. Tìm hoặc tải KuaiSearch-Lite

Notebook ưu tiên dữ liệu đã attach trong `/kaggle/input`. Nếu chưa có, bật Internet để tải từ Hugging Face. `HF_TOKEN` là tùy chọn.

In [3]:
HF_REPO_ID = "benchen4395/KuaiSearch"
TEMP_RAW_ROOT = (
    Path("/kaggle/working/kuaisearch-raw")
    if Path("/kaggle/working").exists()
    else Path.cwd() / "kuaisearch-raw"
).resolve()
RAW_FILES = {
    "items": Path("items_lite/train.jsonl"),
    "users": Path("users_lite/train.jsonl"),
}
RANKING_FILE = Path("rank_lite/train.jsonl")


def has_raw_files(root):
    return root is not None and all((Path(root) / path).is_file() for path in RAW_FILES.values())


if RAW_ROOT is not None:
    RAW_ROOT = Path(RAW_ROOT).expanduser().resolve()
elif Path("/kaggle/input").exists():
    matches = list(Path("/kaggle/input").glob("**/items_lite/train.jsonl"))
    RAW_ROOT = matches[0].parents[1] if matches else None

if not has_raw_files(RAW_ROOT):
    RAW_ROOT = TEMP_RAW_ROOT
    snapshot_download(
        repo_id=HF_REPO_ID,
        repo_type="dataset",
        local_dir=str(RAW_ROOT),
        allow_patterns=[path.as_posix() for path in RAW_FILES.values()],
        token=os.environ.get("HF_TOKEN"),
    )

if not has_raw_files(RAW_ROOT):
    raise FileNotFoundError("KuaiSearch-Lite download is incomplete.")

if OUTPUT_ROOT is None:
    OUTPUT_ROOT = (
        Path("/kaggle/working/preprocessed")
        if Path("/kaggle/working").exists()
        else Path.cwd() / "preprocessed"
    )
OUTPUT_ROOT = Path(OUTPUT_ROOT).expanduser().resolve()

if OUTPUT_ROOT.exists():
    if OUTPUT_ROOT.name != "preprocessed":
        raise ValueError(f"Refusing to reset unsafe path: {OUTPUT_ROOT}")
    shutil.rmtree(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

local_ranking = RAW_ROOT / RANKING_FILE
RANKING_SOURCE = (
    str(local_ranking)
    if local_ranking.is_file()
    else f"hf://datasets/{HF_REPO_ID}/{RANKING_FILE.as_posix()}"
)

print("RAW_ROOT:", RAW_ROOT)
print("RANKING_SOURCE:", RANKING_SOURCE)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

RAW_ROOT: /kaggle/working/kuaisearch-raw
RANKING_SOURCE: hf://datasets/benchen4395/KuaiSearch/rank_lite/train.jsonl
OUTPUT_ROOT: /kaggle/working/preprocessed


## 3. Đọc dữ liệu

In [4]:
items = pl.read_ndjson(RAW_ROOT / RAW_FILES["items"], low_memory=True).to_pandas()
users = pl.read_ndjson(RAW_ROOT / RAW_FILES["users"], low_memory=True).to_pandas()
ranking = pl.scan_ndjson(
    RANKING_SOURCE,
    infer_schema_length=10_000,
    low_memory=True,
)

print(f"Items: {len(items):,}")
print(f"Users: {len(users):,}")
print("Ranking is ready for streaming.")
print(ranking.collect_schema())

Items: 6,634,118
Users: 102,086
Ranking is ready for streaming.
Schema({'user_id': Int64, 'session_id': Int64, 'user_fan_number': Int64, 'user_follow_number': Int64, 'time_index': Int64, 'search_entrance': String, 'recently_clicked_item_ids': List(Int64), 'recently_purchased_item_ids': List(Int64), 'query': String, 'target_item_id': Int64, 'target_item_price': Float64, 'is_clicked': Int64, 'is_purchased': Int64, 'user_statistical_features': Struct({'user_show_cnt_30d_hist': Float64, 'user_click_cnt_30d_hist': Float64, 'user_order_cnt_30d_hist': Float64, 'user_gmv_30d_hist': Float64}), 'target_item_statistical_features': Struct({'item_show_cnt_30d_hist': Float64, 'item_click_cnt_30d_hist': Float64, 'item_order_cnt_30d_hist': Float64}), 'split': String})


## 4. Product catalog

`normalized_text` ghép title, brand, seller và ba cấp category để dùng ở bước embedding.

In [5]:
items = items.rename(columns={"item_id": "product_id"})
if not items["product_id"].is_unique:
    raise ValueError("Expected one row per product_id in items_lite.")
text_columns = [
    "item_title", "brand_name", "seller_name",
    "category_level1_name", "category_level2_name", "category_level3_name",
]
items[text_columns] = items[text_columns].fillna("").astype(str)
items.insert(0, "product_index", range(len(items)))
items["normalized_text"] = (
    "title: " + items["item_title"]
    + " | brand: " + items["brand_name"]
    + " | seller: " + items["seller_name"]
    + " | category_l1: " + items["category_level1_name"]
    + " | category_l2: " + items["category_level2_name"]
    + " | category_l3: " + items["category_level3_name"]
)

items.to_parquet(OUTPUT_ROOT / "items.parquet", index=False, compression="zstd")

display(items.head(3))

,product_index,product_id,item_title,brand_id,brand_name,seller_id,seller_name,category_level1_id,category_level1_name,category_level2_id,category_level2_name,category_level3_id,category_level3_name,normalized_text
0,0,3359347,【香榭丽舍】高级静奢风法式长款遮肉显瘦进口貂皮大衣,1,金粟兰,1,施施高端皮草工厂店,1,女装,1,皮草,0,UNKNOWN,title: 【香榭丽舍】高级静奢风法式长款遮肉显瘦进口貂皮大衣 | brand: 金粟兰 ...
1,1,5096459,福利金钻绒床边地毯防尘免洗,2,其他/other,2,爱家家居地毯,2,家纺,2,地毯地垫,1,地毯,title: 福利金钻绒床边地毯防尘免洗 | brand: 其他/other | selle...
2,2,5935061,三莎夏季舞蹈鞋女式网面广场舞蹈,3,GUARCI,3,漯河邦杰舞蹈鞋,3,瑜伽舞蹈/健身/体育/护具,3,舞蹈/健美操/体操用品,2,广场舞蹈,title: 三莎夏季舞蹈鞋女式网面广场舞蹈 | brand: GUARCI | selle...


## 5. User profiles

In [6]:
user_columns = [
    "user_id", "gender", "age_bucket",
    "fre_country", "fre_province", "fre_city",
]
users = users[user_columns]
users.to_parquet(OUTPUT_ROOT / "users.parquet", index=False, compression="zstd")

display(users.head(3))

,user_id,gender,age_bucket,fre_country,fre_province,fre_city
0,33407,M,31-40,中国,辽宁,鞍山
1,43714,F,31-40,中国,天津,天津
2,79878,F,24-30,中国,吉林,松原


## 6. Ranking

Giữ nguyên hai nhóm statistical features dạng Struct. `split=test` của KuaiSearch được giữ làm test; các session còn lại được chia train/validation bằng hash ổn định. Toàn bộ ranking được stream thẳng vào một file Parquet, không nạp vào RAM.

In [7]:
session_key = pl.concat_str(
    [pl.col("user_id"), pl.col("session_id")],
    separator="|",
)
is_validation = session_key.hash(seed=SEED) % 100 < VALIDATION_PERCENT

ranking = ranking.with_columns(
    pl.when(pl.col("split") == "test")
    .then(pl.lit("test"))
    .when(is_validation)
    .then(pl.lit("validation"))
    .otherwise(pl.lit("train"))
    .alias("data_split")
)

ranking_path = OUTPUT_ROOT / "ranking.parquet"
ranking.sink_parquet(
    ranking_path,
    compression="zstd",
    maintain_order=False,
    engine="streaming",
)

ranking_counts = (
    pl.scan_parquet(ranking_path)
    .group_by("data_split")
    .len()
    .collect(engine="streaming")
)
RANKING_COUNTS = dict(ranking_counts.iter_rows())
print(ranking_counts.sort("data_split"))

if RAW_ROOT.resolve() == TEMP_RAW_ROOT:
    # Remove only the temporary download created by this notebook.
    shutil.rmtree(RAW_ROOT)
    print("Removed temporary raw download:", RAW_ROOT)

shape: (3, 2)
┌────────────┬──────────┐
│ data_split ┆ len      │
│ ---        ┆ ---      │
│ str        ┆ u32      │
╞════════════╪══════════╡
│ test       ┆ 494271   │
│ train      ┆ 15555071 │
│ validation ┆ 1751562  │
└────────────┴──────────┘
Removed temporary raw download: /kaggle/working/kuaisearch-raw


## 7. Product embedding

Encode `normalized_text` bằng Jina Embeddings, chuẩn hoá L2 và lưu ma trận float16 theo đúng thứ tự `product_index`. Hai file đầu ra được `train_rqvae.py` đọc trực tiếp.

In [8]:
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU accelerator on Kaggle before creating embeddings.")

model = SentenceTransformer(
    EMBEDDING_MODEL,
    trust_remote_code=True,
    device="cuda",
    model_kwargs={"dtype": torch.float16},
)

embeddings = model.encode(
    items["normalized_text"].tolist(),
    batch_size=EMBEDDING_BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
    truncate_dim=EMBEDDING_DIM,
)
embeddings = np.asarray(embeddings, dtype=np.float16)

if embeddings.shape != (len(items), EMBEDDING_DIM):
    raise ValueError(f"Unexpected embedding shape: {embeddings.shape}")
if not np.isfinite(embeddings).all():
    raise ValueError("Embeddings contain NaN or Inf.")

np.save(OUTPUT_ROOT / "global_product_embeddings.f16.npy", embeddings)
items[["product_index", "product_id"]].to_parquet(
    OUTPUT_ROOT / "global_embedding_index.parquet", index=False, compression="zstd"
)

sample = embeddings[:min(10_000, len(embeddings))].astype(np.float32)
print("Embedding shape:", embeddings.shape)
print("Embedding dtype:", embeddings.dtype)
print("Mean L2 norm:", np.linalg.norm(sample, axis=1).mean())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/275 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.33k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.36k [00:00<?, ?B/s]

configuration_eurobert.py:   0%|          | 0.00/12.1k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/312 [00:00<?, ?B/s]

Default prompt name is set to 'document'. This prompt will be applied to all inference calls, except if a `prompt` or `prompt_name` parameter is provided.


Batches:   0%|          | 0/51830 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Embedding shape: (6634118, 256)
Embedding dtype: float16
Mean L2 norm: 1.0000044


## 8. Manifest và kiểm tra output

In [9]:
manifest = {
    "contract_version": "kuaisearch-lite-ranking-v1",
    "source": {
        "repo_id": HF_REPO_ID,
        "raw_root": str(RAW_ROOT),
        "ranking": RANKING_SOURCE,
    },
    "configuration": {
        "validation_percent": VALIDATION_PERCENT,
        "seed": SEED,
    },
    "embedding": {
        "model": EMBEDDING_MODEL,
        "dimension": EMBEDDING_DIM,
        "dtype": "float16",
        "normalized": True,
    },
    "products": len(items),
    "users": len(users),
    "ranking": {
        "train": RANKING_COUNTS.get("train", 0),
        "validation": RANKING_COUNTS.get("validation", 0),
        "test": RANKING_COUNTS.get("test", 0),
    },
}
(OUTPUT_ROOT / "preprocessing_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

display(pl.scan_parquet(ranking_path).head(3).collect().to_pandas())
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("Preprocessing complete:", OUTPUT_ROOT)

,user_id,session_id,user_fan_number,user_follow_number,time_index,search_entrance,recently_clicked_item_ids,recently_purchased_item_ids,query,target_item_id,target_item_price,is_clicked,is_purchased,user_statistical_features,target_item_statistical_features,split,data_split
0,1,1,9,65,384565,mall,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 3...",广场舞服装女高档洋气,58,5990.0,1,0,"{'user_show_cnt_30d_hist': 0.0, 'user_click_cn...","{'item_show_cnt_30d_hist': 952.0, 'item_click_...",train,train
1,1,1,9,65,384565,mall,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 3...",广场舞服装女高档洋气,63,4290.0,1,0,"{'user_show_cnt_30d_hist': 0.0, 'user_click_cn...","{'item_show_cnt_30d_hist': 1.0, 'item_click_cn...",train,train
2,1,1,9,65,384565,mall,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 3...",广场舞服装女高档洋气,76,4200.0,1,0,"{'user_show_cnt_30d_hist': 0.0, 'user_click_cn...","{'item_show_cnt_30d_hist': 2431.0, 'item_click...",train,train


{
  "contract_version": "kuaisearch-lite-ranking-v1",
  "source": {
    "repo_id": "benchen4395/KuaiSearch",
    "raw_root": "/kaggle/working/kuaisearch-raw",
    "ranking": "hf://datasets/benchen4395/KuaiSearch/rank_lite/train.jsonl"
  },
  "configuration": {
    "validation_percent": 10,
    "seed": 2026
  },
  "embedding": {
    "model": "jinaai/jina-embeddings-v5-text-nano-clustering",
    "dimension": 256,
    "dtype": "float16",
    "normalized": true
  },
  "products": 6634118,
  "users": 102086,
  "ranking": {
    "train": 15555071,
    "validation": 1751562,
    "test": 494271
  }
}
Preprocessing complete: /kaggle/working/preprocessed


## Artifact đầu ra

- Product catalog: `items.parquet`.
- Product embedding: `global_product_embeddings.f16.npy`, `global_embedding_index.parquet`.
- Ranking cho teacher: `ranking.parquet`, chọn train/validation/test bằng cột `data_split`.
- User features: `users.parquet`.